# Tutorial 03: Core Phenotyping Workflow (Morphological Profiling)

**Before you start:** Run from the repo (notebook under `Notebooks/tutorials/`). This tutorial is the main programmatic entry point for **Morphological Profiling** (often called **Cell Painting** when using specific stains).

**In plain terms:** The model turns each image into an **embedding** (a numeric summary of appearance). **PCA** plots help you see whether treatments separate in that summary space. **Distance to a reference** (e.g. control) quantifies how “far” each sample is from your chosen baseline. By combining these deep embeddings with classical **hand-crafted properties** (like cell size or intensity), we get a complete biological profile.

This notebook covers:
1. Core pipeline (Find, Process, Properties, Distances)
2. Automated **HTML report** generation
3. Basic **Explainability** (Component-Property Correlations)
4. Saving results and exporting configurations

See [Getting Started](../../docs/getting-started.md) and [Common Workflows](../../docs/examples/workflows.md) for more.

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1.1. Optional: Setup Toy Dataset (E. coli)

If you don't have your own data yet, you can download a sample dataset of E. coli images treated with different drugs.

In [ ]:
import requests
import zipfile
from pathlib import Path

def download_toy_dataset(output_dir="./toy_dataset"):
    output_path = Path(output_dir)
    if (output_path / "Ecoli_images").exists():
        print(f"Dataset already exists at {output_path}")
        return output_path

    url = "https://upvehueus-my.sharepoint.com/:u:/g/personal/aitor_gonzalezm_ehu_eus/IQCBYEL7E_-NTZZeIsBhwwG8AbI9BucCBAgaPacYqXjDsA4?download=1"
    output_path.mkdir(exist_ok=True, parents=True)
    zip_path = output_path / "ecoli_dataset.zip"
    
    print("Downloading E. coli dataset...")
    r = requests.get(url, allow_redirects=True)
    with open(zip_path, "wb") as f:
        f.write(r.content)
    
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_path)
    zip_path.unlink()
    
    # Find the actual images dir (might be nested)
    extracted_images = list(output_path.glob("**/Ecoli_images"))
    return extracted_images[0].parent if extracted_images else output_path

# Uncomment to download
# dataset_root = download_toy_dataset()
# image_dir = dataset_root / "Ecoli_images"
# mask_dir = dataset_root / "Ecoli_masks"

## 1. Setup and Reproducibility

Use `PhenoMe(seed=42)` for reproducibility. For full experiment tracking, use `export_experiment_config()` after `save_results()`.

In [ ]:
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

## 2. Path-Based Metadata Extraction

For folder structure `.../(condition)/(timepoint)/(filename).*` (e.g., `data/Treated/24h/cell_001.tif`), use `get_metadata_from_path`.

In [ ]:
# Template: .../(condition)/(timepoint)/(filename).*
metadata_fn = get_metadata_from_path(".../(condition)/(timepoint)/(filename).*")
file_df = pheno.find_files("path/to/images", metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")
print(f"Metadata keys: {file_df.columns.tolist()}")

## 3. Load Model and Process Images

Use `channel_mode='split'` for fluorescence images (each channel processed independently). Use `'combined'` for brightfield/RGB. See [Channel Modes](../../docs/concepts.md#channel-modes).

In [ ]:
# Optional: audit before processing
# audit_df = pheno.audit_data()

model, wrapper = load_dinov2_model(model_name="dinov2_vitg14_reg", device=device)
# Note: You can use `checkpoint_path="my_checkpoint.h5"`
# for regular backups of embeddings during extraction for large datasets.
pheno.process_images(wrapper, channel_mode="split")

## 4. Distance Analysis and Visualization (Optional)

Compute distances to a reference group (e.g. all samples vs `condition='Control'`). **Reading the plots:** points that cluster together are similar in embedding space; the distance distribution shows how far each group lies from the reference centroid.

Use `get_embeddings()` for lazy-safe access when using checkpoints (not shown here). See [Distances](../../docs/reference/api/distances.md) and [Visualization](../../docs/reference/api/visualization.md).

In [ ]:
# Compute distances to reference group
dist_results = pheno.compute_reference_distances(
    reference_filters={"condition": "Control"}, source="embeddings", mode="centroid"
)

# Visualize
pheno.plot_pca(color_by="condition")
pheno.plot_distance_distribution(distance_results=dist_results, group_by="condition")

## 5. Morphological Properties (Cell Painting Style)

**Why extract hand-crafted properties?** While deep embeddings catch subtle patterns, classical features (area, eccentricity, intensity) are easier to interpret biological terms. In **Cell Painting** workflows, we typically extract properties from both the image and a corresponding **mask**.

Use `property_preset='basic'` or `'full'`. See [Property Reference](../../docs/guides/property-reference.md) for preset details.

In [ ]:
# Requires masks (mask_dir in find_files) for shape and masked-intensity features
df_props = pheno.compute_properties(property_preset="basic")
print(f"Computed {len(df_props.columns)} properties for {len(df_props)} samples")

## 6. Basic Explainability: Component-Property Correlations

**What is explainability?** Deep learning models extract abstract embeddings. To understand what biological features (e.g., cell size, intensity) drive the embedding space, we correlate these embeddings (or their PCA/UMAP components) with measured properties. This answers: *"What does this axis in my plot represent?"*

In [ ]:
# Relate PCA components to measured properties
corr_results = pheno.compute_component_correlation(
    method="pca",
    n_components=3,
    top_k=10
)

if corr_results and "summary" in corr_results:
    print("Top property correlations for PC1:")
    display(corr_results["summary"][corr_results["summary"]["Component"] == corr_results["component_names"][0]].head(5))
else:
    print("No correlations found (did you run compute_properties first?)")

## 7. Save Results and Generate Report

Save checkpoint and optionally generate an HTML report. 

**Clustering in the report:** With `include_clustering=True`, the report runs `compute_clustering` internally if no `cluster` labels exist yet. For **interactive** notebooks or scripts—tuning `n_clusters`, comparing to metadata, or using `find_prototypes` / `detect_outliers`—see [06_exploratory_analysis_and_explainability.ipynb](06_exploratory_analysis_and_explainability.ipynb).

In [ ]:
# 1. Save all embeddings and properties to an HDF5-based results structure
pheno.save_results(output_dir="results/")

# Example: To load results later in a new session:
# pheno_new = PhenoMe(device=device, seed=42)
# pheno_new.find_files("path/to/images") # Map paths
# pheno_new.load_results("results/phenome_results.h5") 

# 2. Generate an HTML report
pheno.generate_report(
    output_path="results/report.html",
    reference_filters={"condition": "Control"},
    include_clustering=True,
    include_image_gallery=True,
    color_by="condition",
)

## Next Steps

- [04_advanced_metadata_handling.ipynb](04_advanced_metadata_handling.ipynb): Handling complex CSVs and multi-channel data
- [05_exploratory_analysis_and_explainability.ipynb](05_exploratory_analysis_and_explainability.ipynb): Advanced clustering, prototypes, and deep explainability
- [06_extending_phenome_plugins.ipynb](06_extending_phenome_plugins.ipynb): Writing custom plugins and model wrappers